# 智能论文助手 (PaperAssistant)

## 📝 项目简介

基于 HelloAgents 框架的多智能体论文助手，支持文献检索、论文总结、引用生成、论文润色和大纲生成。

### 作者信息
- 姓名: chengH425
- GitHub: @chengH425
- 日期: 2026-07-20

---
## 第1部分：环境配置

In [ ]:
# 安装依赖（如已安装可跳过）
# !pip install -q hello-agents python-dotenv

In [ ]:
# 导入核心库
import os
import sys
import json
from datetime import datetime
from typing import Dict, Any, List

# Windows 控制台 UTF-8 编码兼容
sys.stdout.reconfigure(encoding='utf-8')

from dotenv import load_dotenv
from hello_agents import (
    HelloAgentsLLM, SimpleAgent, ReflectionAgent,
    PlanSolveAgent, ToolRegistry, Config
)
from hello_agents.tools import Tool, ToolParameter, ToolResponse, ToolStatus

# 加载环境变量
load_dotenv()

print("环境配置完成！")
print(f"   LLM Model: {os.getenv('LLM_MODEL_ID', 'Qwen/Qwen2.5-72B-Instruct')}")

---
## 第2部分：工具定义

In [ ]:
# 从 src/ 模块导入自定义工具（9 个工具）
from src.citation_tool import CitationTool
from src.literature_tool import LiteratureSearchTool
from src.aminer_tool import AminerSearchTool
from src.openalex_tool import OpenAlexSearchTool
from src.pubmed_tool import PubMedSearchTool
from src.crossref_tool import CrossRefSearchTool
from src.arxiv_tool import ArxivSearchTool
from src.pdf_tool import PDFExtractTool

# 文本统计分析工具（轻量级，直接定义）
class TextAnalysisTool(Tool):
    """文本统计分析工具"""
    def __init__(self):
        super().__init__(
            name="text_analysis",
            description="分析文本的统计信息：字数、段落数、句子数等。"
        )
    def run(self, parameters: Dict[str, Any]) -> ToolResponse:
        text = parameters.get("text", "")
        if not text:
            return ToolResponse.error(code="INVALID_PARAM", message="文本不能为空")
        chinese_chars = sum(1 for c in text if '一' <= c <= '鿿')
        english_words = len([w for w in text.split() if any(c.isalpha() for c in w)])
        sentences_cn = len([s for s in text.replace('!', '。').replace('?', '。').split('。') if s.strip()])
        sentences_en = len([s for s in text.replace('!', '.').replace('?', '.').split('.') if s.strip()])
        paragraphs = len([p for p in text.split('\n') if p.strip()])
        result = {
            "总字符数": len(text), "中文字符数": chinese_chars,
            "英文单词数": english_words, "句子数(中)": sentences_cn,
            "句子数(英)": sentences_en, "段落数": paragraphs,
            "预估阅读时间(分钟)": round((chinese_chars / 400 + english_words / 200), 1)
        }
        return ToolResponse.success(
            text=json.dumps(result, ensure_ascii=False, indent=2), data=result)
    def get_parameters(self) -> List[ToolParameter]:
        return [ToolParameter(name="text", type="string",
                             description="要分析的文本内容", required=True)]

print("工具定义完成！共 9 个工具：")
print("   检索类（6个）: Semantic Scholar / AMiner / OpenAlex / PubMed / CrossRef / arXiv")
print("   处理类（3个）: CitationTool / PDFExtractTool / TextAnalysisTool")

---
## 第3部分：智能体构建

本系统使用 **4 种智能体范式** 协同工作：

| 智能体 | 范式 | 职责 |
|--------|------|------|
| SearchAgent | SimpleAgent | 文献检索与信息整理 |
| SummaryAgent | SimpleAgent | 论文内容总结 |
| PolishAgent | ReflectionAgent | 论文润色（自我反思迭代优化） |
| OutlineAgent | PlanSolveAgent | 论文大纲结构化生成 |

In [ ]:
# 创建 LLM 实例（禁用 trace 日志以避免 Windows 编码问题）
llm = HelloAgentsLLM()
config = Config(trace_enabled=False)

# 创建工具注册表，注册全部 9 个工具
tool_registry = ToolRegistry()
tool_registry.register_tool(LiteratureSearchTool())   # Semantic Scholar
tool_registry.register_tool(AminerSearchTool())       # AMiner（中文论文）
tool_registry.register_tool(OpenAlexSearchTool())     # OpenAlex
tool_registry.register_tool(PubMedSearchTool())       # PubMed
tool_registry.register_tool(CrossRefSearchTool())     # CrossRef
tool_registry.register_tool(ArxivSearchTool())        # arXiv
tool_registry.register_tool(CitationTool())
tool_registry.register_tool(PDFExtractTool())
tool_registry.register_tool(TextAnalysisTool())

# ========================================
# 智能体1: 文献检索助手 (SimpleAgent + 6 大检索工具)
# ========================================
search_system_prompt = """你是一位学术文献检索专家。你有 6 个检索工具可用：

- literature_search: Semantic Scholar，全学科覆盖（推荐首选）
- aminer_search: AMiner，中文学术论文（中文文献首选）
- openalex_search: OpenAlex，开放获取论文
- pubmed_search: PubMed，生物医学领域
- crossref_search: CrossRef，期刊论文元数据
- arxiv_search: arXiv，CS/数学/物理预印本

规则：
1. 必须使用用户指定的检索工具获取真实数据
2. 工具调用失败时直接报告错误，不要编造论文
3. 基于真实结果进行分析和推荐"""

search_agent = SimpleAgent(
    name="文献检索助手", llm=llm,
    system_prompt=search_system_prompt, config=config
)
for name in ["literature_search", "aminer_search", "openalex_search",
             "pubmed_search", "crossref_search", "arxiv_search"]:
    search_agent.add_tool(tool_registry.get_tool(name))

# ========================================
# 智能体2: 论文总结助手 (SimpleAgent)
# ========================================
summary_system_prompt = """你是一位学术论文审稿专家，擅长快速提取论文的核心信息。

对于给定的论文内容，请按以下结构生成总结报告：

## 论文信息
- 标题、作者、发表年份、期刊/会议

## 研究问题
- 该论文要解决什么核心问题？

## 方法与创新点
- 采用了什么方法/模型/算法？
- 相比已有工作，核心创新是什么？

## 实验与结果
- 在哪些数据集上做了实验？
- 主要实验结果和性能指标

## 贡献与局限
- 论文的主要贡献（1-3点）
- 论文的局限性或未解决的问题

## 启发与延伸
- 这篇论文对你的研究方向有什么启发？
- 有哪些可以进一步探索的方向？

请使用中文输出报告，专业术语保留英文。"""

summary_agent = SimpleAgent(
    name="论文总结助手",
    llm=llm,
    system_prompt=summary_system_prompt,
    config=config
)

# ========================================
# 智能体3: 论文润色助手 (SimpleAgent，多轮对话模式)
# ========================================
# 在 Web UI 中通过 create_polish_agent() 工厂函数创建独立实例
# 每次新对话创建新 Agent，内部 history 自然累积上下文
# 规则：保持原意、优化表达、记住上文修改历史

# ========================================
# 智能体4: 大纲生成助手 (SimpleAgent，多轮对话模式)
# ========================================
# 在 Web UI 中通过 create_outline_agent() 工厂函数创建独立实例
# 每次新对话创建新 Agent，支持持续细化调整
# 规则：结构化分解、支持"细化第三章"等后续指令

# ========================================
# 智能体5: 论文写作助手 (SimpleAgent + 6 大检索工具，多轮对话模式)
# ========================================
# 在 Web UI 中通过 create_paper_writer_agent() 工厂函数创建独立实例
# 注册全部 6 个检索工具，确保引用真实文献
# 规则：根据大纲逐章撰写、引用前必须先检索、杜绝编造文献
# 支持"写第二章"、"加入更多transformer相关讨论"等后续指令

print("智能体创建完成！")
print("   1. search_agent  - 文献检索助手 (SimpleAgent + 6 检索工具)")
print("   2. summary_agent - 论文总结助手 (SimpleAgent)")
print("   3. polish_agent  - 论文润色助手 (对话模式, Web UI)")
print("   4. outline_agent - 大纲生成助手 (对话模式, Web UI)")
print("   5. writer_agent  - 论文写作助手 (对话模式 + 检索工具, Web UI)")

---
## 第4部分：功能演示

### 📚 演示1：文献检索

In [ ]:
print("=" * 60)
print("📚 演示1：文献检索")
print("=" * 60)

search_query = "大语言模型在软件工程中的应用研究进展"
print(f"\n🔍 检索主题：{search_query}\n")

search_result = search_agent.run(search_query)
print(search_result)
print("\n" + "=" * 60)

### 📝 演示2：论文总结

In [ ]:
print("=" * 60)
print("📝 演示2：论文总结")
print("=" * 60)

# 示例论文摘要
sample_paper = """
论文标题: Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks
作者: Patrick Lewis, Ethan Perez, Aleksandra Piktus, et al.
发表: NeurIPS 2020

摘要: Large pre-trained language models have been shown to store factual knowledge 
in their parameters, and achieve state-of-the-art results when fine-tuned on 
downstream NLP tasks. However, their ability to access and precisely manipulate 
knowledge is still limited, leading to factual errors and hallucinations. We 
introduce Retrieval-Augmented Generation (RAG), a general-purpose fine-tuning 
approach that combines pre-trained parametric and non-parametric memory for 
language generation. RAG models retrieve relevant documents from a dense vector 
index and condition the generation on both the input and retrieved documents. 
We evaluate RAG on a diverse set of NLP tasks including open-domain QA, abstractive 
question answering, and fact verification, achieving state-of-the-art results. 
Our analysis shows that RAG generates more specific, diverse, and factual language 
compared to parametric-only models.
"""

print(f"\n📄 待总结论文：{sample_paper[:80]}...\n")

summary_result = summary_agent.run(
    f"请对以下论文内容进行结构化总结：\n\n{sample_paper}"
)
print(summary_result)
print("\n" + "=" * 60)

### 📎 演示3：引用生成

In [ ]:
print("=" * 60)
print("📎 演示3：多格式引用生成")
print("=" * 60)

# 测试论文数据
test_papers = [
    {
        "title": "Attention Is All You Need",
        "authors": "Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, L., Polosukhin, I.",
        "journal": "Advances in Neural Information Processing Systems",
        "year": "2017",
        "volume": "30",
        "pages": "5998-6008"
    },
    {
        "title": "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding",
        "authors": "Devlin, J., Chang, M. W., Lee, K., Toutanova, K.",
        "journal": "Proceedings of the 2019 Conference of the North American Chapter of the ACL",
        "year": "2019",
        "volume": "1",
        "pages": "4171-4186"
    },
    {
        "title": "Chain-of-Thought Prompting Elicits Reasoning in Large Language Models",
        "authors": "Wei, J., Wang, X., Schuurmans, D., Bosma, M., Ichter, B., Xia, F., Chi, E., Le, Q., Zhou, D.",
        "journal": "Advances in Neural Information Processing Systems",
        "year": "2022",
        "volume": "35",
        "pages": "24824-24837"
    }
]

formats = ["gbt7714", "apa", "mla"]
format_names = {"gbt7714": "GB/T 7714 (中文标准)", "apa": "APA 7th", "mla": "MLA 9th"}

for i, paper in enumerate(test_papers, 1):
    print(f"\n--- 论文 {i}：{paper['title'][:50]}... ---")
    for fmt in formats:
        paper["format"] = fmt
        response = tool_registry.execute_tool("citation_generator", json.dumps(paper))
        print(f"\n  [{format_names[fmt]}]:")
        print(f"  {response.text}")

print("\n" + "=" * 60)

### ✍️ 演示4：论文润色

In [ ]:
print("=" * 60)
print("✍️ 演示4：多轮对话式论文润色")
print("=" * 60)

# 模拟多轮对话
round1 = """请润色以下学术段落，使其更符合学术写作规范：

In this paper, we propose a new method to solve the problem. Our method is very 
good and it works better than other methods. We did a lot of experiments."""

print(f"\n📝 【第1轮】用户: {round1[:80]}...\n")

# 第一轮：初始润色
polish_agent1 = SimpleAgent(
    name="润色", llm=llm, config=config,
    system_prompt="你是学术论文语言编辑。润色后给出修改说明。"
)
result1 = polish_agent1.run(f"用户: {round1}\n助手: ")
print(result1[:500])
print("\n---\n")

# 第二轮：基于上下文继续优化
round2 = "把第二句改得更学术化，使用更精确的词汇"
print(f"📝 【第2轮】用户: {round2}\n")
polish_agent2 = SimpleAgent(
    name="润色", llm=llm, config=config,
    system_prompt="你是学术论文语言编辑。记住上下文，在已有基础上继续修改。"
)
result2 = polish_agent2.run(
    f"用户: {round1}\n助手: {result1}\n用户: {round2}\n助手: "
)
print(result2[:500])
print("\n" + "=" * 60)

### 📊 演示5：论文大纲生成

In [ ]:
print("=" * 60)
print("📊 演示5：论文大纲生成（PlanSolveAgent）")
print("=" * 60)

outline_topic = "基于大语言模型的多智能体协作系统的设计与实现"
print(f"\n📋 论文主题：{outline_topic}\n")
print("🔄 PlanSolveAgent 正在拆解任务并生成大纲...\n")

outline_result = outline_agent.run(
    f"请为以下论文主题生成一份详细的结构化大纲：{outline_topic}"
)
print(outline_result)
print("\n" + "=" * 60)

### 📝 演示6：论文写作（基于大纲 + 真实文献）

展示如何根据大纲逐章撰写论文，写作过程中调用检索工具引用真实文献。

In [ ]:
print("=" * 60)
print("📝 演示6：论文写作（基于大纲 + 真实文献）")
print("=" * 60)

# 模拟一个论文大纲
outline = """论文大纲：基于深度学习的医学影像分析综述
第一章 引言
第二章 医学影像与深度学习基础
第三章 基于CNN的医学影像分析方法
第四章 实验对比与性能评估
第五章 未来展望与挑战"""

print(f"\n📋 给定大纲：\n{outline}\n")
print("🔄 创建论文写作智能体（带文献检索能力）...\n")

# 创建带检索工具的写作智能体
writer_llm = HelloAgentsLLM()
writer_agent = SimpleAgent(
    name="论文写作", llm=writer_llm, config=config,
    system_prompt="""你是学术论文写作专家。可以调用文献检索工具查找真实论文。
引用文献时必须基于检索结果，绝对禁止编造论文。"""
)
# 注册检索工具，确保引用真实文献
for name in ["literature_search", "aminer_search"]:
    writer_agent.add_tool(tool_registry.get_tool(name))

# 第一轮：写引言
result1 = writer_agent.run(
    f"根据以下大纲，请撰写第一章引言部分（约300字）。"
    f"如果需要引用文献，请使用 literature_search 工具搜索真实论文：\n{outline}"
)
print("--- 第一章 引言 ---")
print(result1[:600])
print("...\n")

# 第二轮：写方法部分，需要引用文献
result2 = writer_agent.run(
    "请撰写第三章内容，介绍至少两种主流的基于CNN的医学影像分析方法。"
    "请使用 aminer_search 或 literature_search 工具搜索相关论文，并在正文中引用。"
)
print("--- 第三章 基于CNN的方法 ---")
print(result2[:600])
print("...\n")

print("=" * 60)

---
## 第5部分：文本分析工具演示

In [ ]:
print("=" * 60)
print("📈 附加演示：文本统计分析")
print("=" * 60)

sample_text = """
大语言模型（Large Language Models, LLMs）在近年来取得了突破性进展。
以GPT系列为代表的预训练语言模型在自然语言处理任务中展现出强大的能力。
然而，LLMs在实际应用中仍面临幻觉问题、推理能力不足等挑战。
本文综述了近年来关于LLM推理能力增强的研究进展，
重点分析了Chain-of-Thought、Tree-of-Thought等提示方法的原理和效果。
研究表明，结构化的推理路径设计能显著提升LLM在复杂推理任务上的表现。
We systematically review recent advances in enhancing LLM reasoning capabilities.
Our analysis covers both prompting-based methods and training-based approaches.
The results demonstrate that structured reasoning significantly improves performance.
"""

response = tool_registry.execute_tool("text_analysis", json.dumps({"text": sample_text}))
print(response.text)
print("\n" + "=" * 60)

---
## 第6部分：总结与展望

### ✅ 已实现的功能

1. **文献检索** — 基于 SimpleAgent 的智能文献检索助手，提供系统化的检索策略
2. **论文总结** — 结构化提取论文核心信息（问题、方法、贡献、局限）
3. **引用生成** — 自定义 CitationTool 支持 GB/T 7714 / APA / MLA 三种格式
4. **论文润色** — 基于 ReflectionAgent 的自我反思迭代优化
5. **大纲生成** — 基于 PlanSolveAgent 的结构化论文大纲

### 🔧 技术亮点

- **多范式智能体协作**：融合了 SimpleAgent、ReflectionAgent、PlanSolveAgent 三种范式
- **自定义工具系统**：基于 Tool + ToolParameter 实现了 CitationTool 和 TextAnalysisTool
- **结构化输出**：每个功能都有清晰的 Markdown 格式输出

### 🚧 遇到的挑战

- LLM API 延迟波动，需通过 tool 端确定性计算弥补
- 引用格式规则复杂，APA/MLA 的边缘情况需要进一步细化

### 🔮 未来改进方向

- [ ] 接入 arXiv API 实现实时论文检索
- [ ] 支持 PDF 上传与解析
- [ ] 增加论文查重分析功能
- [ ] 构建 Gradio Web 界面
- [ ] 引入多智能体辩论机制提升审稿质量